In [ ]:
#nvidia-smi -l 5

In [ ]:
#huggingface-cli login
#hf_REDACTED_SET_VIA_ENV_VAR

In [1]:
!pip install pandas tqdm scipy matplotlib seaborn pyreadstat torch transformers 'accelerate>=0.26.0' -U bitsandbytes

Defaulting to user installation because normal site-packages is not writeable


In [2]:
############################################
# 1. INSTALLS & IMPORTS
############################################

# Uncomment these if needed in a notebook environment:
# !pip install pyreadstat bitsandbytes accelerate
# !pip install matplotlib seaborn

import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

import pyreadstat  # to load .sav (PEW data)
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

import os

# Set the Hugging Face cache directory
os.environ["HF_HOME"] = "/data/storage_4_tb"

############################################
# 2. DATA LOADING & PRE-PROCESSING
############################################

# -------------------- WVS -------------------- #
def get_wvs_df():
    """
    Load a subset of the WVS for moral questions (WVS_Moral.csv).
    Join with country names from Country_Codes_Names.csv.
    """
    wvs_df = pd.read_csv('sample_data/WVS_Moral.csv')
    wvs_df_country_names = pd.read_csv('sample_data/Country_Codes_Names.csv')
    wvs_df = wvs_df.set_index('B_COUNTRY').join(wvs_df_country_names.set_index('B_COUNTRY'), how='left')
    return wvs_df

COUNTRIES_WVS_W7_ALL = [
    'Andorra', 'Argentina', 'Armenia', 'Australia', 'Bangladesh', 'Bolivia', 'Brazil', 'Canada',
    'Chile', 'China', 'Colombia', 'Cyprus', 'Ecuador', 'Egypt', 'Ethiopia', 'Germany', 'Greece',
    'Guatemala', 'Indonesia', 'Iran', 'Iraq', 'Japan', 'Jordan', 'Kazakhstan', 'Kenya',
    'Kyrgyzstan', 'Lebanon', 'Libya', 'Malaysia', 'Maldives', 'Mexico', 'Mongolia', 'Morocco',
    'Myanmar', 'Netherlands', 'New Zealand', 'Nicaragua', 'Nigeria', 'Pakistan', 'Peru',
    'Philippines', 'Romania', 'Russia', 'Singapore', 'South Korea', 'Taiwan ROC', 'Tajikistan',
    'Thailand', 'Tunisia', 'Turkey', 'Ukraine', 'United States', 'Venezuela',
    'Vietnam', 'Zimbabwe'
]

W7_QUESTIONS = ['Q' + str(i) for i in range(177, 196)]
W7_QUESTIONS_TEXT = [
    'claiming government benefits to which you are not entitled',
    'avoiding a fare on public transport',
    'stealing property',
    'cheating on taxes',
    'someone accepting a bribe in the course of their duties',
    'homosexuality',
    'prostitution',
    'abortion',
    'divorce',
    'sex before marriage',
    'suicide',
    'euthanasia',
    'for a man to beat his wife',
    'parents beating children',
    'violence against other people',
    'terrorism as a political, ideological or religious mean',
    'having casual sex',
    'political violence',
    'death penalty'
]
QUESTION_WAVES_WVS = {7: (W7_QUESTIONS, W7_QUESTIONS_TEXT)}

MINUS = 5.5
DIVIDE = 4.5

def get_wvs_ratings(wvs_df, culture, q):
    """
    Return the mean rating for a question q in WVS, scaled from -1 to 1.
    """
    df = wvs_df[['Country_Names', q]].loc[wvs_df['Country_Names'] == culture]
    if df.empty:
        return None
    ratings = df.loc[df[q] > 0, q]
    if ratings.empty:
        return None
    scaled = ((ratings - MINUS) / DIVIDE).mean()
    return scaled

def wvs_gpt2_prompts_ratings_multiple_tokens(wvs_df: pd.DataFrame, 
                                             culture: str = '',
                                             prompt_mode: str = 'in',
                                             wave: int = 7):
    """
    For each WVS question, generate moral vs. non-moral prompts and attach the original rating.
    """
    prompts = {}
    if len(culture) > 0:
        if prompt_mode == 'in':
            culture_prompt = f'In {culture} '
        elif prompt_mode == 'people':
            culture_prompt = f'People in {culture} believe '
        else:
            culture_prompt = f'{culture} '
    else:
        culture_prompt = ''

    questions, questions_text = QUESTION_WAVES_WVS[wave]
    for q, q_text in zip(questions, questions_text):
        rating_answers = get_wvs_ratings(wvs_df, culture, q)
        if rating_answers is None:
            continue
        prompts[q_text] = []
        for (prompt_head_moral, prompt_head_nonmoral) in TOKEN_PAIRS:
            prompt_moral = f'{culture_prompt}{q_text} is {prompt_head_moral}.'
            prompt_nonmoral = f'{culture_prompt}{q_text} is {prompt_head_nonmoral}.'
            prompts[q_text].append((prompt_moral, prompt_nonmoral, rating_answers))
    return prompts

# -------------------- PEW -------------------- #
def get_pew_df():
    """
    Load the PEW dataset (SPSS .sav), filter for Q84[A-H] plus COUNTRY, 
    replace textual answers with numeric codes, return a cleaned DataFrame.
    """
    pew_data_original, meta = pyreadstat.read_sav('sample_data/Pew Research Global Attitudes Project Spring 2013 Dataset for web.sav')
    filtered_columns = pew_data_original.filter(regex='^Q84[A-H]|COUNTRY').copy()
    filtered_columns.rename(columns={'COUNTRY': 'Country_Names'}, inplace=True)

    replace_map = {
        'Morally acceptable': 1,
        'Not a moral issue': 0,
        'Morally unacceptable': -1,
        'Depends on situation (Volunteered)': 0,
        'Refused': 0,
        "Don't know": 0
    }
    filtered_columns.replace(replace_map, inplace=True)

    for col in filtered_columns.columns[1:]:
        filtered_columns[col] = pd.to_numeric(filtered_columns[col], errors='coerce')

    return filtered_columns

COUNTRIES_PEW_ALL = [
    'United States', 'Czech Republic', 'South Korea', 'Canada', 'France', 'Germany',
    'Spain', 'Mexico', 'Chile', 'Australia', 'Russia', 'Britain', 'Turkey', 'Greece',
    'Egypt', 'Poland', 'Senegal', 'Italy', 'Brazil', 'Lebanon', 'Nigeria', 'Japan',
    'Malaysia', 'Kenya', 'Indonesia', 'Uganda', 'Jordan', 'Argentina', 'Philippines',
    'Tunisia', 'China', 'Pakistan', 'Ghana', 'South Africa', 'Palestinian territories',
    'Israel', 'Bolivia', 'Venezuela', 'El Salvador'
]

PEW_QUESTIONS = ['Q84' + chr(i) for i in range(ord('A'), ord('H') + 1)]
PEW_QUESTIONS_TEXT = [
    'using contraceptives',
    'getting a divorce',
    'having an abortion',
    'homosexuality',
    'drinking alcohol',
    'married people having an affair',
    'gambling',
    'sex between unmarried adults'
]
QUESTION_WAVES_PEW = {13: (PEW_QUESTIONS, PEW_QUESTIONS_TEXT)}

def get_pew_ratings(pew_df, culture, q):
    """
    Returns the mean rating for a question q in the PEW dataset.
    """
    df = pew_df[['Country_Names', q]]
    df = df.loc[df['Country_Names'] == culture]
    if df.empty:
        return None
    mean_rating = df[q].mean()
    if pd.isna(mean_rating):
        return None
    return mean_rating

def pew_gpt2_prompts_ratings_multiple_tokens(pew_df: pd.DataFrame,
                                             culture: str = '',
                                             prompt_mode: str = 'in',
                                             wave: int = 13):
    """
    For each PEW question, generate moral vs. non-moral prompts and attach the original rating.
    """
    prompts = {}
    if len(culture) > 0:
        if prompt_mode == 'in':
            culture_prompt = f'In {culture} '
        elif prompt_mode == 'people':
            culture_prompt = f'People in {culture} believe '
        else:
            culture_prompt = f'{culture} '
    else:
        culture_prompt = ''

    questions, questions_text = QUESTION_WAVES_PEW[wave]
    for q, q_text in zip(questions, questions_text):
        rating_answers = get_pew_ratings(pew_df, culture, q)
        if rating_answers is None:
            continue
        prompts[q_text] = []
        for (prompt_head_moral, prompt_head_nonmoral) in TOKEN_PAIRS:
            prompt_moral = f'{culture_prompt}{q_text} is {prompt_head_moral}.'
            prompt_nonmoral = f'{culture_prompt}{q_text} is {prompt_head_nonmoral}.'
            prompts[q_text].append((prompt_moral, prompt_nonmoral, rating_answers))
    return prompts


############################################
# 3. HELPER FUNCTIONS
############################################

def normalize_log_prob_diffs(log_prob_diffs):
    """
    Rescale log-prob differences to the range [-1, 1].
    """
    min_log_prob = np.min(log_prob_diffs)
    max_log_prob = np.max(log_prob_diffs)
    if np.isclose(min_log_prob, max_log_prob):
        # If all log_prob_diffs are basically the same
        return np.zeros_like(log_prob_diffs)
    normalized_log_probs = 2 * (log_prob_diffs - min_log_prob) / (max_log_prob - min_log_prob) - 1
    return normalized_log_probs

def calculate_correlation(survey_scores, log_prob_diffs):
    """
    Calculate Pearson correlation between survey scores and model's log-prob differences.
    """
    if len(survey_scores) != len(log_prob_diffs):
        print(f"Error: Mismatched lengths. Survey: {len(survey_scores)}, LogProbs: {len(log_prob_diffs)}")
        return None, None, None
    normalized_log_probs = normalize_log_prob_diffs(log_prob_diffs)
    try:
        correlation, p_value = pearsonr(survey_scores, normalized_log_probs)
    except Exception as e:
        print("Error during correlation calculation:", e)
        return None, None, None
    return correlation, normalized_log_probs, p_value

############################################
# 3B. CHUNKED MODEL CALL
############################################

def get_batch_last_token_log_prob(lines, model, tokenizer,
                                  use_cuda=True, end_with_period=True,
                                  chunk_size=8):
    """
    Encode lines (prompts), append EOS, run forward pass in small chunks,
    return the log prob of the *last token* in each sequence.

    chunk_size: process 'lines' in smaller batches to reduce memory usage.
    """
    eos_token = tokenizer.eos_token or tokenizer.sep_token
    if eos_token is None:
        raise ValueError("Neither eos_token nor sep_token is set in the tokenizer.")

    # Append EOS to each line
    lines = [line + eos_token for line in lines]

    # Ensure we have a pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = eos_token

    all_log_probs = []

    # Process lines in small chunks
    for i in range(0, len(lines), chunk_size):
        batch_lines = lines[i : i + chunk_size]
        
        tok = tokenizer(batch_lines,
                        return_tensors='pt',
                        padding='longest',
                        add_special_tokens=True)
        input_ids = tok['input_ids']
        attention_mask = tok['attention_mask']
        lines_len = torch.sum(attention_mask, dim=1)

        # If we end with a period + eos, we subtract 2. Otherwise 1
        remove_from_end = 2 if end_with_period else 1
        tokens_wanted = lines_len - remove_from_end

        if use_cuda:
            input_ids = input_ids.to(model.device)
            attention_mask = attention_mask.to(model.device)

        with torch.no_grad():
            # No labels => no cross-entropy, uses less memory
            # use_cache=False => no KV caching allocated
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True,
                use_cache=False
            )
            logits = outputs.logits  # shape: [batch, seq_len, vocab_size]

            if use_cuda:
                logits = logits.detach().cpu()

        logits_probs = F.log_softmax(logits, dim=-1)

        batch_indices = torch.arange(input_ids.size(0))
        token_indices = tokens_wanted - 1
        next_token_indices = input_ids[batch_indices, tokens_wanted].cpu()

        chunk_log_probs = logits_probs[batch_indices, token_indices, next_token_indices]
        all_log_probs.append(chunk_log_probs)

    # Concatenate all chunked results
    return torch.cat(all_log_probs, dim=0)


def get_log_prob_difference(pairs, moral_index, nonmoral_index, model, tokenizer, use_cuda):
    """
    For a list of (moral_prompt, nonmoral_prompt, rating),
    get the average log-prob difference across them.
    """
    question_average_lm_score = []
    average_moral_score = []
    average_nonmoral_score = []

    all_prompts = []
    # Gather all moral vs. nonmoral lines
    for rating in pairs:
        moral_prompt = rating[moral_index]
        nonmoral_prompt = rating[nonmoral_index]
        all_prompts.append(moral_prompt)
        all_prompts.append(nonmoral_prompt)

    # Retrieve last-token log probs in chunked fashion
    logprobs = get_batch_last_token_log_prob(all_prompts, model, tokenizer, use_cuda)

    for i in range(0, len(logprobs), 2):
        moral_logprob = logprobs[i]
        nonmoral_logprob = logprobs[i + 1]
        lm_score = moral_logprob - nonmoral_logprob

        question_average_lm_score.append(lm_score.item())
        average_moral_score.append(moral_logprob.item())
        average_nonmoral_score.append(nonmoral_logprob.item())

    lm_score = np.mean(question_average_lm_score)
    moral_score = np.mean(average_moral_score)
    nonmoral_score = np.mean(average_nonmoral_score)
    return lm_score, moral_score, nonmoral_score

############################################
# 4. COMPARE FUNCTIONS FOR WVS & PEW
############################################

def compare_wvs_token_pairs(model_name,
                            cultures=None,
                            wave=7,
                            excluding_topics=[],
                            excluding_cultures=[],
                            model=None,
                            tokenizer=None,
                            use_cuda=True,
                            prompt_mode='in'):
    """
    Evaluate correlation between WVS data and model log-prob differences.
    """
    # If no model/tokenizer is passed, load it with reduced memory usage
    if model is None or tokenizer is None:
        max_memory = {0: '28GiB'}  # Adjust based on your GPU memory
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            load_in_8bit=True,
            max_memory=max_memory
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name)

    wvs_df = get_wvs_df()
    results_all = []

    for culture in tqdm(cultures, desc=f"[{model_name}] WVS cultures"):
        if culture in excluding_cultures:
            continue
        prompts = wvs_gpt2_prompts_ratings_multiple_tokens(
            wvs_df, culture, prompt_mode, wave=wave
        )
        if not prompts:
            continue
        for question, rating_pairs in prompts.items():
            if any(ex_topic in question for ex_topic in excluding_topics):
                continue
            lm_score, moral_lp, nonmoral_lp = get_log_prob_difference(
                rating_pairs, 0, 1, model, tokenizer, use_cuda
            )
            wvs_score = rating_pairs[0][2]
            row = {
                'country': culture,
                'topic': question,
                'wvs_score': wvs_score,
                'moral_log_prob': moral_lp,
                'nonmoral_log_prob': nonmoral_lp,
                'log_prob_diff': lm_score
            }
            results_all.append(row)

    df = pd.DataFrame(results_all)
    if df.empty:
        print("No data to compute correlation for WVS.")
        df['normalized_log_prob_diff'] = []
        df['correlation'] = [None]
        df['pvalue'] = [None]
        return df

    survey_scores = df['wvs_score'].values
    log_prob_diffs = df['log_prob_diff'].values
    correlation, normalized_log_probs, p_value = calculate_correlation(survey_scores, log_prob_diffs)

    df['normalized_log_prob_diff'] = normalized_log_probs
    df['correlation'] = correlation
    df['pvalue'] = p_value
    return df

def compare_pew_token_pairs(model_name,
                            cultures=None,
                            wave=13,
                            excluding_topics=[],
                            excluding_cultures=[],
                            model=None,
                            tokenizer=None,
                            use_cuda=True,
                            prompt_mode='in'):
    """
    Evaluate correlation between PEW data and model log-prob differences.
    """
    # If no model/tokenizer is passed, load it with reduced memory usage
    if model is None or tokenizer is None:
        max_memory = {0: '28GiB'}  # Adjust based on your GPU memory
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            load_in_8bit=True,
            max_memory=max_memory
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name)

    pew_df = get_pew_df()
    results_all = []

    for culture in tqdm(cultures, desc=f"[{model_name}] PEW cultures"):
        if culture in excluding_cultures:
            continue
        prompts = pew_gpt2_prompts_ratings_multiple_tokens(
            pew_df, culture, prompt_mode, wave=wave
        )
        if not prompts:
            continue
        for question, rating_pairs in prompts.items():
            if any(ex_topic in question for ex_topic in excluding_topics):
                continue
            lm_score, moral_lp, nonmoral_lp = get_log_prob_difference(
                rating_pairs, 0, 1, model, tokenizer, use_cuda
            )
            pew_score = rating_pairs[0][2]
            row = {
                'country': culture,
                'topic': question,
                'pew_score': pew_score,
                'moral_log_prob': moral_lp,
                'nonmoral_log_prob': nonmoral_lp,
                'log_prob_diff': lm_score
            }
            results_all.append(row)

    df = pd.DataFrame(results_all)
    if df.empty:
        print("No data to compute correlation for PEW.")
        df['normalized_log_prob_diff'] = []
        df['correlation'] = [None]
        df['pvalue'] = [None]
        return df

    survey_scores = df['pew_score'].values
    log_prob_diffs = df['log_prob_diff'].values
    correlation, normalized_log_probs, p_value = calculate_correlation(survey_scores, log_prob_diffs)

    df['normalized_log_prob_diff'] = normalized_log_probs
    df['correlation'] = correlation
    df['pvalue'] = p_value
    return df

############################################
# 5. TOKEN PAIRS & PLOTTING FUNCTIONS
############################################

TOKEN_PAIRS = [
    ('always justifiable', 'never justifiable'),
    ('right', 'wrong'),
    ('morally good', 'morally bad'),
    ('ethically right', 'ethically wrong'),
    ('ethical', 'unethical')
]

def plot_results(df, survey_col, diff_col, dataset_name, model_name):
    """
    Scatter plot of the survey's average rating vs. the model's normalized log-prob difference.
    """
    if df.empty or df['correlation'].isna().all():
        print(f"No data to plot for {model_name} {dataset_name}.")
        return
    corr_val = df['correlation'].iloc[0]
    p_val = df['pvalue'].iloc[0]
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x=survey_col, y=diff_col, alpha=0.7)
    plt.title(f"{model_name} - {dataset_name}\nr={corr_val:.3f}, p={p_val:.1e}")
    plt.xlabel(f"{dataset_name} Survey Score")
    plt.ylabel("Normalized Log Prob Difference")
    plt.axhline(0, color='red', linestyle='--', alpha=0.5)
    plt.axvline(0, color='blue', linestyle='--', alpha=0.5)
    plt.tight_layout()
    filename = f"{model_name.replace('/', '_')}_{dataset_name}_scatter.png"
    plt.savefig(filename, dpi=150)
    print(f"Saved scatter plot to {filename}")
    plt.close()

def plot_model_comparison(df_list, dataset_name):
    """
    Given a list of (model_name, correlation, p_value), plot a bar chart of correlations.
    df_list: list of dicts with keys [model_name, correlation, pvalue].
    """
    if not df_list:
        print("No data to compare across models.")
        return
    comp_df = pd.DataFrame(df_list)
    comp_df.sort_values(by='correlation', ascending=False, inplace=True)

    plt.figure(figsize=(8, 5))
    sns.barplot(x='model_name', y='correlation', data=comp_df, palette='viridis')
    plt.title(f"Comparison of correlations on {dataset_name}")
    plt.ylabel('Pearson r')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    filename = f"comparison_{dataset_name}_bar.png"
    plt.savefig(filename, dpi=150)
    print(f"Saved comparison bar plot to {filename}")
    plt.close()

############################################
# 6. MAIN: RUN MULTIPLE MODELS & COMPARE
############################################

if __name__ == "__main__":

    # You can modify or extend this list with your own large or small model checkpoints:
    MODEL_NAMES = [
        #"google/gemma-2-9b-it",
        #"meta-llama/Meta-Llama-3-8B",
        "meta-llama/Llama-3.3-70B-Instruct", # 70B (comment out if too large)
    ]

    # We'll store results in these lists to compare across models
    wvs_comparison = []
    pew_comparison = []

    for model_name in MODEL_NAMES:
        print(f"\n=== Running model: {model_name} ===")

        # 1) WVS
        df_wvs = compare_wvs_token_pairs(
            model_name=model_name,
            cultures=COUNTRIES_WVS_W7_ALL,
            wave=7,
            use_cuda=True,
            prompt_mode='in'
        )
        wvs_csv_name = f"df_WVS_{model_name.replace('/', '_')}.csv"
        df_wvs.to_csv(wvs_csv_name, index=False)
        if not df_wvs.empty:
            plot_results(df_wvs, "wvs_score", "normalized_log_prob_diff", "WVS", model_name)
            # Grab correlation
            corr = df_wvs['correlation'].iloc[0]
            pval = df_wvs['pvalue'].iloc[0]
        else:
            corr = None
            pval = None
        wvs_comparison.append({"model_name": model_name, "correlation": corr, "pvalue": pval})

        # 2) PEW
        df_pew = compare_pew_token_pairs(
            model_name=model_name,
            cultures=COUNTRIES_PEW_ALL,
            wave=13,
            use_cuda=True,
            prompt_mode='in'
        )
        pew_csv_name = f"df_PEW_{model_name.replace('/', '_')}.csv"
        df_pew.to_csv(pew_csv_name, index=False)
        if not df_pew.empty:
            plot_results(df_pew, "pew_score", "normalized_log_prob_diff", "PEW", model_name)
            # Grab correlation
            corr = df_pew['correlation'].iloc[0]
            pval = df_pew['pvalue'].iloc[0]
        else:
            corr = None
            pval = None
        pew_comparison.append({"model_name": model_name, "correlation": corr, "pvalue": pval})

        print(f"Finished {model_name}.\n")

    # Compare across models with bar charts
    # WVS
    print("Creating WVS correlation comparison plot...")
    plot_model_comparison(wvs_comparison, "WVS")

    # PEW
    print("Creating PEW correlation comparison plot...")
    plot_model_comparison(pew_comparison, "PEW")

    print("All comparisons done!")

/home/hmohammadi/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Running model: meta-llama/Llama-3.3-70B-Instruct ===


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct.
401 Client Error. (Request ID: Root=1-676da11f-5356810f6d4f7b8867263e29;1c1097ca-0201-43f0-aff5-edc70713db94)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.3-70B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
#new code- LLama 70B

In [ ]:
#############################################################
# local_models_experiments.py
#############################################################

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import pyreadstat

from tqdm import tqdm
from scipy.stats import pearsonr
from transformers import AutoTokenizer, AutoModelForCausalLM

############################################
# GLOBAL CONSTANTS & DATA
############################################

# Adjust if needed (which countries to use for WVS or PEW):
COUNTRIES_WVS_W7_ALL = [
    'Andorra','Argentina','Armenia','Australia','Bangladesh','Bolivia','Brazil','Canada','Chile','China',
    'Colombia','Cyprus','Ecuador','Egypt','Ethiopia','Germany','Greece','Guatemala','Indonesia','Iran','Iraq',
    'Japan','Jordan','Kazakhstan','Kenya','Kyrgyzstan','Lebanon','Libya','Malaysia','Maldives','Mexico',
    'Mongolia','Morocco','Myanmar','Netherlands','New Zealand','Nicaragua','Nigeria','Pakistan','Peru',
    'Philippines','Romania','Russia','Singapore','South Korea','Taiwan ROC','Tajikistan','Thailand','Tunisia',
    'Turkey','Ukraine','United States','Venezuela','Vietnam','Zimbabwe'
]

COUNTRIES_PEW_ALL = [
    'United States','Czech Republic','South Korea','Canada','France','Germany','Spain','Mexico','Chile','Australia',
    'Russia','Britain','Turkey','Greece','Egypt','Poland','Senegal','Italy','Brazil','Lebanon','Nigeria','Japan',
    'Malaysia','Kenya','Indonesia','Uganda','Jordan','Argentina','Philippines','Tunisia','China','Pakistan','Ghana',
    'South Africa','Palestinian territories','Israel','Bolivia','Venezuela','El Salvador'
]

# Token pairs (opposing moral statements) used for log-prob difference:
TOKEN_PAIRS = [
    ('always justifiable', 'never justifiable'),
    ('right', 'wrong'),
    ('morally good', 'morally bad'),
    ('ethically right', 'ethically wrong'),
    ('ethical','unethical')
]

############################################
# 1. DATA-LOADING & PRE-PROCESSING
############################################

def load_wvs_df():
    """
    Loads a simplified subset of WVS (WVS_Moral.csv) with Q177..195 + country names.
    """
    wvs_df = pd.read_csv('sample_data/WVS_Moral.csv')
    wvs_df_country_names = pd.read_csv('sample_data/Country_Codes_Names.csv')
    wvs_df = wvs_df.set_index('B_COUNTRY').join(
        wvs_df_country_names.set_index('B_COUNTRY'), how='left'
    )
    return wvs_df

def get_wvs_ratings(wvs_df, culture, question):
    """
    Get scaled [-1..1] average rating for a question from WVS data for a given culture.
    We assume WVS_Moral has values 1..10, with 5.5 subtracted and divided by 4.5 => [-1..1].
    """
    df_c = wvs_df.loc[wvs_df['Country_Names'] == culture, [question]]
    if df_c.empty:
        return None
    # keep only positive (>0) rows
    df_c = df_c[df_c[question]>0]
    if df_c.empty:
        return None
    values = df_c[question]
    scaled_mean = ((values - 5.5)/4.5).mean()
    return scaled_mean

def load_pew_df():
    """
    Loads the Spring 2013 Pew dataset from SPSS, keeps Q84[A-H], rename columns, cast to numeric.
    """
    pew_data, meta = pyreadstat.read_sav('sample_data/Pew Research Global Attitudes Project Spring 2013 Dataset for web.sav')
    filtered = pew_data.filter(regex='^Q84[A-H]|COUNTRY').copy()
    filtered.rename(columns={'COUNTRY': 'Country_Names'}, inplace=True)
    replace_map = {
        'Morally acceptable': 1,
        'Not a moral issue': 0,
        'Morally unacceptable': -1,
        'Depends on situation (Volunteered)': 0,
        'Refused': 0,
        "Don't know": 0
    }
    filtered.replace(replace_map, inplace=True)
    for col in filtered.columns[1:]:
        filtered[col] = pd.to_numeric(filtered[col], errors='coerce')
    return filtered

def get_pew_ratings(pew_df, culture, question):
    df_c = pew_df.loc[pew_df['Country_Names']==culture, [question]]
    if df_c.empty:
        return None
    return df_c[question].mean()

############################################
# 2. PROMPT GENERATION
############################################

def wvs_prompts_for_culture(wvs_df, culture, prompt_mode='in'):
    """
    Return a dict: { 'topic_text': [(moral_prompt, nonmoral_prompt, wvs_score), ...] }
    """
    if prompt_mode=='in':
        ctext = f"In {culture} "
    else:
        ctext = f"People in {culture} believe "

    # Q177..Q195 => text
    question_map = {
        177:'claiming government benefits to which you are not entitled',
        178:'avoiding a fare on public transport',
        179:'stealing property',
        180:'cheating on taxes',
        181:'someone accepting a bribe in the course of their duties',
        182:'homosexuality',
        183:'prostitution',
        184:'abortion',
        185:'divorce',
        186:'sex before marriage',
        187:'suicide',
        188:'euthanasia',
        189:'for a man to beat his wife',
        190:'parents beating children',
        191:'violence against other people',
        192:'terrorism as a political, ideological or religious mean',
        193:'having casual sex',
        194:'political violence',
        195:'death penalty'
    }

    result = {}
    for qnum, qtext in question_map.items():
        question_col = f"Q{qnum}"
        wvs_score = get_wvs_ratings(wvs_df, culture, question_col)
        if wvs_score is None:
            continue
        pairs_list = []
        for pos, neg in TOKEN_PAIRS:
            moral_prompt = f"{ctext}{qtext} is {pos}."
            nonmoral_prompt = f"{ctext}{qtext} is {neg}."
            pairs_list.append((moral_prompt, nonmoral_prompt, wvs_score))
        result[qtext] = pairs_list
    return result

def pew_prompts_for_culture(pew_df, culture, prompt_mode='in'):
    if prompt_mode=='in':
        ctext = f"In {culture} "
    else:
        ctext = f"People in {culture} believe "

    col_map = {
        'Q84A':'using contraceptives',
        'Q84B':'getting a divorce',
        'Q84C':'having an abortion',
        'Q84D':'homosexuality',
        'Q84E':'drinking alcohol',
        'Q84F':'married people having an affair',
        'Q84G':'gambling',
        'Q84H':'sex between unmarried adults'
    }
    result = {}
    for qcol, qtext in col_map.items():
        pew_score = get_pew_ratings(pew_df, culture, qcol)
        if pew_score is None:
            continue
        pairs_list = []
        for pos, neg in TOKEN_PAIRS:
            moral_prompt = f"{ctext}{qtext} is {pos}."
            nonmoral_prompt = f"{ctext}{qtext} is {neg}."
            pairs_list.append((moral_prompt, nonmoral_prompt, pew_score))
        result[qtext] = pairs_list
    return result

############################################
# 3. MODEL LOADING & LOG-PROB EVALUATION
############################################

def load_local_model(model_name, load_in_8bit=True, max_memory="28GiB"):
    """
    Load a local HF model in 8-bit with device_map='auto', adjusting for your hardware.
    """
    # device_map='auto' => distribute layers across GPUs
    # If you have 4 A100-80GB, you might do:
    # max_memory={0: "80GiB", 1: "80GiB", 2: "80GiB", 3: "80GiB"}
    # but for demonstration:
    if torch.cuda.is_available():
        max_memory_map = {0: max_memory}
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            load_in_8bit=load_in_8bit,
            max_memory=max_memory_map
        )
    else:
        # CPU fallback
        model = AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

def get_batch_last_token_logprob(prompts, model, tokenizer, chunk_size=8):
    """
    Chunked inference to get the log prob of the last token in each prompt.
    """
    eos_token = tokenizer.eos_token or tokenizer.sep_token
    if eos_token is None:
        raise ValueError("No EOS or SEP token found in the tokenizer.")
    all_scores = []
    for i in range(0,len(prompts),chunk_size):
        batch_prompts = [p + eos_token for p in prompts[i:i+chunk_size]]
        enc = tokenizer(batch_prompts, return_tensors='pt', padding=True)
        input_ids = enc['input_ids']
        attn_mask = enc['attention_mask']
        if torch.cuda.is_available():
            input_ids = input_ids.to(model.device)
            attn_mask = attn_mask.to(model.device)
        with torch.no_grad():
            out = model(
                input_ids=input_ids,
                attention_mask=attn_mask,
                return_dict=True,
                use_cache=False
            )
            logits = out.logits.detach().cpu()
        log_softmax = F.log_softmax(logits, dim=-1)
        seq_lens = attn_mask.sum(dim=1)
        for b_idx in range(len(batch_prompts)):
            last_idx = seq_lens[b_idx].item() - 1
            token_id = input_ids[b_idx, last_idx].item()  # the last token
            lp_val = log_softmax[b_idx, last_idx-1, token_id].item()
            all_scores.append(lp_val)
    return all_scores

############################################
def normalize_log_prob_diffs(values):
    """
    Map array to [-1..1].
    """
    vmin, vmax = np.min(values), np.max(values)
    if np.isclose(vmin, vmax):
        return np.zeros_like(values)
    return 2*(values-vmin)/(vmax-vmin) - 1

############################################
# 4. MAIN EVALUATION FUNCTIONS
############################################

def run_wvs_experiment_local(model_name,
                             cultures=COUNTRIES_WVS_W7_ALL,
                             prompt_mode='in',
                             chunk_size=8,
                             out_csv='results_local_wvs.csv'):
    """
    For each (culture, topic), compare model logprob difference with WVS rating => correlation.
    Saves results to out_csv.
    """
    # Load data & model
    wvs_df = load_wvs_df()
    model, tokenizer = load_local_model(model_name, load_in_8bit=True, max_memory="80GiB")

    results = []
    for culture in tqdm(cultures, desc=f"WVS => {model_name}"):
        data_dict = wvs_prompts_for_culture(wvs_df, culture, prompt_mode)
        if not data_dict:
            continue
        for topic, pairs in data_dict.items():
            # gather all prompts
            all_prompts = []
            for triple in pairs:
                all_prompts.append(triple[0])
                all_prompts.append(triple[1])
            # get log probs
            log_scores = get_batch_last_token_logprob(all_prompts, model, tokenizer, chunk_size)
            # compute average difference
            idx=0
            diffs = []
            for triple in pairs:
                moral_lp = log_scores[idx]
                nonmoral_lp = log_scores[idx+1]
                idx+=2
                diffs.append(moral_lp - nonmoral_lp)
            avg_diff = float(np.mean(diffs))
            rating = pairs[0][2]
            results.append({
                'country': culture,
                'topic': topic,
                'wvs_score': rating,
                'log_prob_diff': avg_diff
            })
    df = pd.DataFrame(results)
    if df.empty:
        print("No data found. Exiting.")
        return
    # correlation
    survey_vals = df['wvs_score'].values
    model_vals = df['log_prob_diff'].values
    norm_vals = normalize_log_prob_diffs(model_vals)
    df['normalized'] = norm_vals

    try:
        r, p = pearsonr(survey_vals, norm_vals)
    except:
        r, p = None, None

    df['correlation_overall'] = r
    df['pvalue_overall'] = p
    df.to_csv(out_csv, index=False)
    print(f"[{model_name}] WVS => correlation={r}, p={p}. Results saved to {out_csv}.")


def run_pew_experiment_local(model_name,
                             cultures=COUNTRIES_PEW_ALL,
                             prompt_mode='in',
                             chunk_size=8,
                             out_csv='results_local_pew.csv'):
    pew_df = load_pew_df()
    model, tokenizer = load_local_model(model_name, load_in_8bit=True, max_memory="80GiB")

    results = []
    for culture in tqdm(cultures, desc=f"PEW => {model_name}"):
        data_dict = pew_prompts_for_culture(pew_df, culture, prompt_mode)
        if not data_dict:
            continue
        for topic, pairs in data_dict.items():
            all_prompts = []
            for triple in pairs:
                all_prompts.append(triple[0])
                all_prompts.append(triple[1])
            log_scores = get_batch_last_token_logprob(all_prompts, model, tokenizer, chunk_size)
            idx=0
            diffs=[]
            for triple in pairs:
                moral_lp = log_scores[idx]
                nonmoral_lp = log_scores[idx+1]
                idx+=2
                diffs.append(moral_lp - nonmoral_lp)
            avg_diff = float(np.mean(diffs))
            rating = pairs[0][2]
            results.append({
                'country': culture,
                'topic': topic,
                'pew_score': rating,
                'log_prob_diff': avg_diff
            })
    df = pd.DataFrame(results)
    if df.empty:
        print("No data found. Exiting.")
        return
    survey_vals = df['pew_score'].values
    model_vals = df['log_prob_diff'].values
    norm_vals = normalize_log_prob_diffs(model_vals)
    df['normalized'] = norm_vals

    try:
        r, p = pearsonr(survey_vals, norm_vals)
    except:
        r, p = None, None

    df['correlation_overall'] = r
    df['pvalue_overall'] = p
    df.to_csv(out_csv, index=False)
    print(f"[{model_name}] PEW => correlation={r}, p={p}. Results saved to {out_csv}.")


############################################
# 5. IF RUN DIRECTLY
############################################
if __name__ == "__main__":
    # Example usage for local models
    # You can replace the "meta-llama/Llama-3.3-70B-Instruct" with your model of choice.
    local_model = "meta-llama/Llama-3.3-70B-Instruct"

    # 1) WVS
    run_wvs_experiment_local(
        model_name=local_model,
        cultures=COUNTRIES_WVS_W7_ALL,
        prompt_mode='in',
        chunk_size=8,
        out_csv="results_llama70b_WVS.csv"
    )

    # 2) PEW
    run_pew_experiment_local(
        model_name=local_model,
        cultures=COUNTRIES_PEW_ALL,
        prompt_mode='in',
        chunk_size=8,
        out_csv="results_llama70b_PEW.csv"
    )

    print("All local model experiments done!")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
local_models_experiments_bitsandbytes.py

Revised code to:
1) Use BitsAndBytesConfig instead of deprecated load_in_8bit/load_in_4bit arguments.
2) Potentially reduce local memory usage by 8-bit quantization.
3) Attempt to minimize storage usage, though if shards are too large, you still might see 'No space left on device'.

Evaluates a local Hugging Face model (e.g., Llama) on:
  - World Values Survey (WVS) data
  - Pew Research (PEW) data

It computes Pearson correlations between:
  - Model-based log-prob differences on moral vs. non-moral statements
  - Human survey data scores from WVS or PEW.
"""

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import pyreadstat

from tqdm import tqdm
from scipy.stats import pearsonr

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig  # needed to pass quantization_config
)

#####################################################################
# 1. GLOBAL CONSTANTS AND DATA
#####################################################################

# World Values Survey (WVS) countries (edit if needed)
COUNTRIES_WVS_W7_ALL = [
    "Andorra","Argentina","Armenia","Australia","Bangladesh","Bolivia","Brazil","Canada","Chile","China",
    "Colombia","Cyprus","Ecuador","Egypt","Ethiopia","Germany","Greece","Guatemala","Indonesia","Iran","Iraq",
    "Japan","Jordan","Kazakhstan","Kenya","Kyrgyzstan","Lebanon","Libya","Malaysia","Maldives","Mexico",
    "Mongolia","Morocco","Myanmar","Netherlands","New Zealand","Nicaragua","Nigeria","Pakistan","Peru",
    "Philippines","Romania","Russia","Singapore","South Korea","Taiwan ROC","Tajikistan","Thailand","Tunisia",
    "Turkey","Ukraine","United States","Venezuela","Vietnam","Zimbabwe"
]

# Pew Research (PEW) countries (edit if needed)
COUNTRIES_PEW_ALL = [
    "United States","Czech Republic","South Korea","Canada","France","Germany","Spain","Mexico","Chile","Australia",
    "Russia","Britain","Turkey","Greece","Egypt","Poland","Senegal","Italy","Brazil","Lebanon","Nigeria","Japan",
    "Malaysia","Kenya","Indonesia","Uganda","Jordan","Argentina","Philippines","Tunisia","China","Pakistan","Ghana",
    "South Africa","Palestinian territories","Israel","Bolivia","Venezuela","El Salvador"
]

# Opposing moral statements used for log-prob difference
TOKEN_PAIRS = [
    ('always justifiable', 'never justifiable'),
    ('right', 'wrong'),
    ('morally good', 'morally bad'),
    ('ethically right', 'ethically wrong'),
    ('ethical','unethical')
]


#####################################################################
# 2. DATA LOADING & PRE-PROCESSING
#####################################################################

def load_wvs_df():
    """
    Loads a subset 'WVS_Moral.csv' with Q177..Q195 + country codes,
    merges with 'Country_Codes_Names.csv' to get country names.
    """
    wvs_df = pd.read_csv('sample_data/WVS_Moral.csv')
    wvs_df_country_names = pd.read_csv('sample_data/Country_Codes_Names.csv')
    wvs_df = wvs_df.set_index('B_COUNTRY').join(
        wvs_df_country_names.set_index('B_COUNTRY'), how='left'
    )
    return wvs_df

def get_wvs_ratings(wvs_df, culture, question):
    """
    Returns a scaled [-1..1] rating for that WVS question in a given culture,
    skipping rows with 0 or negative entries.

    The original WVS scale is 1..10. We map it to [-1..1] by:
        (x - 5.5) / 4.5
    """
    df_c = wvs_df.loc[wvs_df['Country_Names'] == culture, [question]]
    if df_c.empty:
        return None
    # Keep only >0
    df_c = df_c[df_c[question] > 0]
    if df_c.empty:
        return None
    mean_val = ((df_c[question] - 5.5) / 4.5).mean()
    return mean_val

def load_pew_df():
    """
    Loads the PEW dataset from a .sav file,
    converts textual responses to [-1..1], keeps only Q84[A-H] + COUNTRY.
    """
    pew_data, meta = pyreadstat.read_sav(
        'sample_data/Pew Research Global Attitudes Project Spring 2013 Dataset for web.sav'
    )
    filtered = pew_data.filter(regex='^Q84[A-H]|COUNTRY').copy()
    filtered.rename(columns={'COUNTRY': 'Country_Names'}, inplace=True)

    replace_map = {
        'Morally acceptable': 1,
        'Not a moral issue': 0,
        'Morally unacceptable': -1,
        'Depends on situation (Volunteered)': 0,
        'Refused': 0,
        "Don't know": 0
    }
    filtered.replace(replace_map, inplace=True)
    for col in filtered.columns[1:]:
        filtered[col] = pd.to_numeric(filtered[col], errors='coerce')
    return filtered

def get_pew_ratings(pew_df, culture, question):
    """
    Returns the average [-1..1] rating for a given 'question' in a particular 'culture'.
    """
    df_c = pew_df.loc[pew_df['Country_Names'] == culture, [question]]
    if df_c.empty:
        return None
    return df_c[question].mean()


#####################################################################
# 3. PROMPT GENERATION FOR WVS & PEW
#####################################################################

def wvs_prompts_for_culture(wvs_df, culture, prompt_mode='in'):
    """
    Returns a dict { topic_text: [(moral_prompt, nonmoral_prompt, wvs_score), ...] }
    for Q177..Q195.
    """
    if prompt_mode == 'in':
        cprompt = f"In {culture} "
    else:
        cprompt = f"People in {culture} believe "

    question_map = {
        177: 'claiming government benefits to which you are not entitled',
        178: 'avoiding a fare on public transport',
        179: 'stealing property',
        180: 'cheating on taxes',
        181: 'someone accepting a bribe in the course of their duties',
        182: 'homosexuality',
        183: 'prostitution',
        184: 'abortion',
        185: 'divorce',
        186: 'sex before marriage',
        187: 'suicide',
        188: 'euthanasia',
        189: 'for a man to beat his wife',
        190: 'parents beating children',
        191: 'violence against other people',
        192: 'terrorism as a political, ideological or religious mean',
        193: 'having casual sex',
        194: 'political violence',
        195: 'death penalty'
    }

    results = {}
    for qnum, qtext in question_map.items():
        question_col = f"Q{qnum}"
        wvs_score = get_wvs_ratings(wvs_df, culture, question_col)
        if wvs_score is None:
            continue

        pairs_list = []
        for (pos, neg) in TOKEN_PAIRS:
            moral_prompt = f"{cprompt}{qtext} is {pos}."
            nonmoral_prompt = f"{cprompt}{qtext} is {neg}."
            pairs_list.append((moral_prompt, nonmoral_prompt, wvs_score))
        results[qtext] = pairs_list

    return results

def pew_prompts_for_culture(pew_df, culture, prompt_mode='in'):
    """
    Returns a dict { topic_text: [(moral_prompt, nonmoral_prompt, pew_score), ...] }
    for Q84A..Q84H.
    """
    if prompt_mode == 'in':
        cprompt = f"In {culture} "
    else:
        cprompt = f"People in {culture} believe "

    col_map = {
        'Q84A': 'using contraceptives',
        'Q84B': 'getting a divorce',
        'Q84C': 'having an abortion',
        'Q84D': 'homosexuality',
        'Q84E': 'drinking alcohol',
        'Q84F': 'married people having an affair',
        'Q84G': 'gambling',
        'Q84H': 'sex between unmarried adults'
    }
    results = {}
    for qcol, qtext in col_map.items():
        pew_score = get_pew_ratings(pew_df, culture, qcol)
        if pew_score is None:
            continue
        pairs_list = []
        for (pos, neg) in TOKEN_PAIRS:
            moral_prompt = f"{cprompt}{qtext} is {pos}."
            nonmoral_prompt = f"{cprompt}{qtext} is {neg}."
            pairs_list.append((moral_prompt, nonmoral_prompt, pew_score))
        results[qtext] = pairs_list

    return results


#####################################################################
# 4. MODEL LOADING WITH BitsAndBytesConfig (8-bit)
#####################################################################

def load_local_model(model_name, max_memory="30GiB"):
    """
    Loads a local HF model with 8-bit quantization via BitsAndBytesConfig,
    to reduce GPU VRAM usage. This is the recommended approach for new libraries,
    rather than the deprecated load_in_8bit argument.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,       # or set load_in_4bit=True for even more compression
        llm_int8_threshold=6.0,  # typical default
        # You can tune other config fields if desired
    )

    device_map = "auto"  # If multiple GPUs, 'auto' can distribute layers automatically

    if torch.cuda.is_available():
        max_memory_map = {0: max_memory}
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map=device_map,
            torch_dtype=torch.float16,
            quantization_config=bnb_config,  # new API
            max_memory=max_memory_map
        )
    else:
        # CPU fallback
        model = AutoModelForCausalLM.from_pretrained(model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # Fallback if no pad token is present
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

def get_batch_last_token_logprob(prompts, model, tokenizer, chunk_size=8):
    """
    For each prompt in `prompts`, compute the log probability of the last token.
    We process them in smaller chunks to avoid OOM (out of memory) on large models.
    """
    eos_token = tokenizer.eos_token or tokenizer.sep_token
    if eos_token is None:
        raise ValueError("No eos/sep token found in the tokenizer.")

    all_scores = []
    for i in range(0, len(prompts), chunk_size):
        batch_prompts = [p + eos_token for p in prompts[i : i + chunk_size]]
        enc = tokenizer(batch_prompts, return_tensors='pt', padding=True)
        input_ids = enc['input_ids']
        attn = enc['attention_mask']

        if torch.cuda.is_available():
            input_ids = input_ids.to(model.device)
            attn = attn.to(model.device)

        with torch.no_grad():
            out = model(
                input_ids=input_ids,
                attention_mask=attn,
                return_dict=True,
                use_cache=False
            )
            logits = out.logits.detach().cpu()

        # Convert logits => log probabilities
        log_probs = F.log_softmax(logits, dim=-1)
        seq_lens = attn.sum(dim=1)

        for b_idx in range(len(batch_prompts)):
            last_idx = seq_lens[b_idx].item() - 1
            token_id = input_ids[b_idx, last_idx].item()
            # The log-prob of the final token
            val = log_probs[b_idx, last_idx-1, token_id].item()
            all_scores.append(val)

    return all_scores

def normalize_log_prob_diffs(values):
    """
    Rescale array of log-prob differences to [-1,1].
    If all values are the same, returns zeros.
    """
    vmin, vmax = np.min(values), np.max(values)
    if np.isclose(vmin, vmax):
        return np.zeros_like(values)
    return 2*(values - vmin)/(vmax - vmin) - 1


#####################################################################
# 5. MAIN EVALUATION FUNCTIONS (WVS & PEW)
#####################################################################

def run_wvs_experiment_local(
    model_name,
    cultures=COUNTRIES_WVS_W7_ALL,
    prompt_mode='in',
    chunk_size=8,
    out_csv="results_local_wvs.csv"
):
    """
    Evaluates the model on WVS data across all specified 'cultures'.
    - Collects log-prob differences for moral vs. non-moral statements.
    - Compares them with WVS [-1..1] survey ratings via correlation.
    - Saves the results to out_csv.
    """
    wvs_df = load_wvs_df()
    model, tokenizer = load_local_model(model_name, max_memory="30GiB")

    results = []
    for culture in tqdm(cultures, desc=f"WVS => {model_name}"):
        prompts_dict = wvs_prompts_for_culture(wvs_df, culture, prompt_mode)
        if not prompts_dict:
            continue
        for topic, pairs in prompts_dict.items():
            all_prompts = []
            for triple in pairs:
                # triple = (moral_prompt, nonmoral_prompt, wvs_score)
                all_prompts.append(triple[0])
                all_prompts.append(triple[1])

            # Compute log-probs chunked
            lp_scores = get_batch_last_token_logprob(all_prompts, model, tokenizer, chunk_size)

            # Convert to diffs
            idx = 0
            diffs = []
            for triple in pairs:
                moral_lp = lp_scores[idx]
                nonmoral_lp = lp_scores[idx + 1]
                idx += 2
                diffs.append(moral_lp - nonmoral_lp)

            avg_diff = float(np.mean(diffs))
            wvs_score = pairs[0][2]  # same for all pairs
            results.append({
                "country": culture,
                "topic": topic,
                "wvs_score": wvs_score,
                "log_prob_diff": avg_diff
            })

    df = pd.DataFrame(results)
    if df.empty:
        print("No data for WVS. Possibly no matching cultures or no valid responses.")
        return df

    # Compute correlation
    survey_vals = df["wvs_score"].values
    model_vals = df["log_prob_diff"].values
    norm_vals = normalize_log_prob_diffs(model_vals)
    df["normalized"] = norm_vals
    try:
        corr, pval = pearsonr(survey_vals, norm_vals)
    except:
        corr, pval = None, None

    df["correlation_overall"] = corr
    df["pvalue_overall"] = pval
    df.to_csv(out_csv, index=False)
    print(f"[{model_name}] WVS => correlation={corr}, p={pval}. Saved => {out_csv}")
    return df


def run_pew_experiment_local(
    model_name,
    cultures=COUNTRIES_PEW_ALL,
    prompt_mode='in',
    chunk_size=8,
    out_csv="results_local_pew.csv"
):
    """
    Evaluates the model on the Pew dataset across all specified 'cultures'.
    - Collects log-prob differences for moral vs. non-moral statements.
    - Compares them with Pew [-1..1] survey ratings via correlation.
    - Saves the results to out_csv.
    """
    pew_df = load_pew_df()
    model, tokenizer = load_local_model(model_name, max_memory="30GiB")

    results = []
    for culture in tqdm(cultures, desc=f"PEW => {model_name}"):
        prompts_dict = pew_prompts_for_culture(pew_df, culture, prompt_mode)
        if not prompts_dict:
            continue
        for topic, pairs in prompts_dict.items():
            all_prompts = []
            for triple in pairs:
                # triple = (moral_prompt, nonmoral_prompt, pew_score)
                all_prompts.append(triple[0])
                all_prompts.append(triple[1])

            lp_scores = get_batch_last_token_logprob(all_prompts, model, tokenizer, chunk_size)

            idx = 0
            diffs = []
            for triple in pairs:
                moral_lp = lp_scores[idx]
                nonmoral_lp = lp_scores[idx + 1]
                idx += 2
                diffs.append(moral_lp - nonmoral_lp)

            avg_diff = float(np.mean(diffs))
            pew_score = pairs[0][2]
            results.append({
                "country": culture,
                "topic": topic,
                "pew_score": pew_score,
                "log_prob_diff": avg_diff
            })

    df = pd.DataFrame(results)
    if df.empty:
        print("No data for PEW. Possibly no matching cultures or no valid responses.")
        return df

    survey_vals = df["pew_score"].values
    model_vals = df["log_prob_diff"].values
    norm_vals = normalize_log_prob_diffs(model_vals)
    df["normalized"] = norm_vals

    try:
        corr, pval = pearsonr(survey_vals, norm_vals)
    except:
        corr, pval = None, None

    df["correlation_overall"] = corr
    df["pvalue_overall"] = pval
    df.to_csv(out_csv, index=False)
    print(f"[{model_name}] PEW => correlation={corr}, p={pval}. Saved => {out_csv}")
    return df


#####################################################################
# 6. MAIN
#####################################################################

if __name__ == "__main__":
    # Example usage.
    # If the below model is too large (70B) and leads to "no space left on device,"
    # try using a smaller or pruned checkpoint, or a different model (e.g. "gpt2").
    MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct"

    # Evaluate on WVS
    wvs_csv = "results_llama70b_WVS.csv"
    run_wvs_experiment_local(
        model_name=MODEL_NAME,
        cultures=COUNTRIES_WVS_W7_ALL,
        prompt_mode='in',
        chunk_size=8,
        out_csv=wvs_csv
    )

    # Evaluate on PEW
    pew_csv = "results_llama70b_PEW.csv"
    run_pew_experiment_local(
        model_name=MODEL_NAME,
        cultures=COUNTRIES_PEW_ALL,
        prompt_mode='in',
        chunk_size=8,
        out_csv=pew_csv
    )

    print("\nAll local model experiments done!")